# Derive a single or set of best performing model configurations for confirmatory analysis

## Set Up

In [1]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
import pandas as pd
from src.perm_res import (
    get_avg_r2,
    get_layer1,
    get_layer2,
    get_layer3,
)

raw_preds_dir = BASE_PATH / "results/perm_res/preds/baseline"
r2_dir = BASE_PATH / "results/comprehensive_r2"
compare_dir = BASE_PATH / "results/compare"

In [2]:
parent_base_df = pd.read_parquet(raw_preds_dir / "parent/pred_df.parquet")
self_base_df = pd.read_parquet(raw_preds_dir / "self/pred_df.parquet")

## Load preds

In [ ]:
r2_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
r2_df_parent = get_avg_r2(parent_base_df)
r2_1_parent = get_layer1(r2_df_parent)
print("1 layer:")
display(r2_1_parent)
# 2
r2s_2_S_base_parent, r2s_2_Q_base_parent = get_layer2(r2_df_parent, filter_best_Q=True)
print("2 layer S base:")
display(r2s_2_S_base_parent)
print("2 layer Q base:")
display(r2s_2_Q_base_parent)
# 3
r2_3_parent = get_layer3(r2_df_parent, filter_best_Q=True)
print("3 layer:")
display(r2_3_parent)
##Export
r2_df_parent.to_parquet(r2_dir / "parent.parquet")

In [ ]:
r2_df_self = get_avg_r2(self_base_df)
# 1
r2_1_self = get_layer1(r2_df_self)
display(r2_1_self)
# 2
r2s_2_S_base_self, r2s_2_Q_base_self = get_layer2(r2_df_self, filter_best_Q=True)
display(r2s_2_S_base_self)
display(r2s_2_Q_base_self)
# 3
r2_3_self = get_layer3(r2_df_self, filter_best_Q=True)
display(r2_3_self)
##Export
r2_df_self.to_parquet(r2_dir / "self.parquet")

## Compare models

Bayesian comparison

In [3]:
parent_r2 = pd.read_parquet(r2_dir / "parent.parquet")
self_r2 = pd.read_parquet(r2_dir / "self.parquet")

In [4]:
params = [
    "num_layers",
    "agg_type",
    "llm_model",
    "desc_seg",
    "alpha",
    "two_layer_base",
    "temp",
]
from sklearn.metrics import mean_absolute_error, r2_score
from baycomp import two_on_single

SCORE_TYPE = "r2"

### Calculate Bayesian Compare Vals- Within SQT

In [ ]:
def export_data(data_to_export, export_path, remove_existing=False):
    if remove_existing and export_path.exists():
        print(f"REMOVING file at path: {export_path}")
        export_path.unlink()
    export_path.parent.mkdir(exist_ok=True, parents=True)
    if export_path.suffix == ".parquet":
        data_to_export.to_parquet(export_path)
    else:
        raise ValueError(f"Unrecognized export type: {export_path.suffix}")


def get_top_models(r2_df, params, top_n=50, num_layers=3, two_layer_base=None):
    if num_layers == 3:
        r2_n_layer = get_layer3(r2_df, filter_best_Q=True)
        print("Running 3 layer...")
    elif num_layers == 2:
        # only consider 2 layer ST (not QT)
        r2_n_layer_S, r2_n_layer_Q = get_layer2(r2_df, filter_best_Q=True)
        if two_layer_base == "Q":
            r2_n_layer = r2_n_layer_Q
        elif two_layer_base == "S":
            r2_n_layer = r2_n_layer_S
        else:
            raise ValueError(
                f"two_layer_base must be one of ['S', 'Q'], got {two_layer_base}"
            )
        print("Running 2 layer...")
    elif num_layers == 1:
        r2_n_layer = get_layer1(r2_df)
        print("Running 1 layer...")

    r2_n_layer = r2_n_layer.sort_values(by="diff", ascending=False)
    # Take top 50 model configs
    r2_top = r2_n_layer.iloc[:top_n]
    top_model_params = r2_top.iloc[0][params]
    return r2_top, top_model_params


def get_score(pred_df, score_type, param_vals=None):
    """
    df: pandas DF
            Contains predictions for all models
    param_vals: pd.Series
            Mapping params to specific values for filtering
    type: str
            One of ['mae', 'r2']
    """
    if param_vals is not None:
        df_sub = pred_df[
            (pred_df[param_vals.keys().to_list()] == param_vals).all(axis=1)
        ]
    else:
        df_sub = pred_df
    if score_type == "mae":
        score_df_fold_repeats = (
            df_sub.groupby(["repeat", "split_id"])
            .apply(lambda g: mean_absolute_error(g["pred_stage3_total"], g["true"]))
            .rename("mae")
            .reset_index()
        )
        score_df_repeats = (
            df_sub.groupby(["repeat"])
            .apply(lambda g: mean_absolute_error(g["pred_stage3_total"], g["true"]))
            .rename("mae")
            .reset_index()
        )
        score_df_fold_repeats_q = (
            df_sub.groupby(["repeat", "split_id"])
            .apply(lambda g: mean_absolute_error(g["pred_stage2_total"], g["true"]))
            .rename("mae")
            .reset_index()
        )
        score_df_repeats_q = (
            df_sub.groupby(["repeat"])
            .apply(lambda g: mean_absolute_error(g["pred_stage2_total"], g["true"]))
            .rename("mae")
            .reset_index()
        )

    elif score_type == "r2":
        # display(df_sub)
        score_df_fold_repeats = (
            df_sub.groupby(["repeat", "split_id"])
            .apply(lambda g: r2_score(y_pred=g["pred_stage3_total"], y_true=g["true"]))
            .rename("r2")
            .reset_index()
        )
        score_df_repeats = (
            df_sub.groupby(["repeat"])
            .apply(lambda g: r2_score(y_pred=g["pred_stage3_total"], y_true=g["true"]))
            .rename("r2")
            .reset_index()
        )
        score_df_fold_repeats_q = (
            df_sub.groupby(["repeat", "split_id"])
            .apply(lambda g: r2_score(y_pred=g["pred_stage2_total"], y_true=g["true"]))
            .rename("r2")
            .reset_index()
        )
        score_df_repeats_q = (
            df_sub.groupby(["repeat"])
            .apply(lambda g: r2_score(y_pred=g["pred_stage2_total"], y_true=g["true"]))
            .rename("r2")
            .reset_index()
        )

    return (
        score_df_fold_repeats,
        score_df_repeats,
        score_df_fold_repeats_q,
        score_df_repeats_q,
    )


def compare_model_configs(
    base_pred_df,
    r2_df,
    param_list,
    top_n_models,
    score_type,
    num_layers=3,
    two_layer_base=None,
):
    r2_n_top, top_model_params = get_top_models(
        r2_df,
        param_list,
        top_n=top_n_models,
        num_layers=num_layers,
        two_layer_base=two_layer_base,
    )
    best_score_df_rf, _, _, _ = get_score(
        pred_df=base_pred_df, param_vals=top_model_params, score_type=score_type
    )

    new_rows = []
    for i, row in r2_n_top.reset_index(drop=True).iterrows():
        new_row = row[params].to_dict()
        cur_score_df_rf, cur_score_df_r, cur_score_df_rf_q, cur_score_df_r_q = (
            get_score(
                pred_df=base_pred_df, param_vals=row[param_list], score_type=score_type
            )
        )
        if i == 0:
            new_row.update(
                {
                    "P(best better)": 0.0,
                    "P(equal)": 1.0,
                    "P(this better)": 0.0,
                }
            )
        else:
            combined_df = pd.merge(
                best_score_df_rf,
                cur_score_df_rf,
                on=["repeat", "split_id"],
                suffixes=("_best", "_cur"),
            )
            # (p_left, p_rope, p_right), fig = two_on_single(
            p_left, p_rope, p_right = two_on_single(
                x=combined_df[f"{score_type}_best"],
                y=combined_df[f"{score_type}_cur"],
                rope=0.01,
                runs=10,
                plot=False,
            )
            new_row.update(
                {
                    "P(best better)": round(p_left, 2),
                    "P(equal)": round(p_rope, 2),
                    "P(this better)": round(p_right, 2),
                }
            )
        # average across n_repeats * n_folds values
        avg_r2_rf = float(cur_score_df_rf[score_type].mean())
        std_r2_rf = float(cur_score_df_rf[score_type].std())
        avg_r2_rf_q = float(cur_score_df_rf_q[score_type].mean())
        std_r2_rf_q = float(cur_score_df_rf_q[score_type].std())
        # average across n_repeats values (each val calculated within a repeat across folds)
        avg_r2_r = float(cur_score_df_r[score_type].mean())
        std_r2_r = float(cur_score_df_r[score_type].std())
        avg_r2_r_q = float(cur_score_df_r_q[score_type].mean())
        std_r2_r_q = float(cur_score_df_r_q[score_type].std())
        new_row.update(
            {
                f"{score_type} repeats-folds": f"{avg_r2_rf:.3f} ({std_r2_rf:.3f})",
                f"{score_type} repeats": f"{avg_r2_r:.3f} ({std_r2_r:.3f})",
                f"{score_type} repeats-folds- Q": f"{avg_r2_rf_q:.3f} ({std_r2_rf_q:.3f})",
                f"{score_type} repeats- Q": f"{avg_r2_r_q:.3f} ({std_r2_r_q:.3f})",
            }
        )
        new_rows.append(new_row)

    compare_df = pd.DataFrame(new_rows)
    return compare_df

#### 3 layer

In [ ]:
compare_df_parent_3 = compare_model_configs(
    base_pred_df=parent_base_df,
    r2_df=parent_r2,
    param_list=params,
    top_n_models=25,
    score_type=SCORE_TYPE,
)
export_data(
    data_to_export=compare_df_parent_3,
    export_path=compare_dir / f"3/parent_{SCORE_TYPE}.parquet",
    remove_existing=True,
)


compare_df_self_3 = compare_model_configs(
    base_pred_df=self_base_df,
    r2_df=self_r2,
    param_list=params,
    top_n_models=25,
    score_type=SCORE_TYPE,
)
export_data(
    data_to_export=compare_df_self_3,
    export_path=compare_dir / f"3/self_{SCORE_TYPE}.parquet",
    remove_existing=True,
)

In [ ]:
# 18 ~equal
display(compare_df_parent_3.sort_values("P(equal)", ascending=False))
# 2 ~equal
display(compare_df_self_3.sort_values("P(equal)", ascending=False))

#### 2 layer

In [ ]:
TWO_LAYER_BASE = "Q"
compare_df_parent_2 = compare_model_configs(
    base_pred_df=parent_base_df,
    r2_df=parent_r2,
    param_list=params,
    top_n_models=25,
    score_type=SCORE_TYPE,
    num_layers=2,
    two_layer_base=TWO_LAYER_BASE,
)
export_data(
    data_to_export=compare_df_parent_2,
    export_path=compare_dir / f"2/parent_{SCORE_TYPE}_{TWO_LAYER_BASE}.parquet",
    remove_existing=True,
)


compare_df_self_2 = compare_model_configs(
    base_pred_df=self_base_df,
    r2_df=self_r2,
    param_list=params,
    top_n_models=25,
    score_type=SCORE_TYPE,
    num_layers=2,
    two_layer_base=TWO_LAYER_BASE,
)
export_data(
    data_to_export=compare_df_self_2,
    export_path=compare_dir / f"2/self_{SCORE_TYPE}_{TWO_LAYER_BASE}.parquet",
    remove_existing=True,
)

In [ ]:
#
display(compare_df_parent_2.sort_values("P(equal)", ascending=False))
#
display(compare_df_self_2.sort_values("P(equal)", ascending=False))

#### 1 layer

In [ ]:
compare_df_parent_1 = compare_model_configs(
    base_pred_df=parent_base_df,
    r2_df=parent_r2,
    param_list=params,
    top_n_models=25,
    score_type=SCORE_TYPE,
    num_layers=1,
)
export_data(
    data_to_export=compare_df_parent_1,
    export_path=compare_dir / f"1/parent_{SCORE_TYPE}.parquet",
    remove_existing=True,
)


compare_df_self_1 = compare_model_configs(
    base_pred_df=self_base_df,
    r2_df=self_r2,
    param_list=params,
    top_n_models=25,
    score_type=SCORE_TYPE,
    num_layers=1,
)
export_data(
    data_to_export=compare_df_self_1,
    export_path=compare_dir / f"1/self_{SCORE_TYPE}.parquet",
    remove_existing=True,
)

In [ ]:
#
display(compare_df_parent_1.sort_values("P(equal)", ascending=False))
#
display(compare_df_self_1.sort_values("P(equal)", ascending=False))

#### Utilize vals for compare/selection

- Via correlation cluster mapping

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


def plot_scatter(error_best, error_cur, i):
    fig, ax = plt.subplots()
    ax.scatter(
        error_best,
        error_cur,
        label="prediction error per participant in each fold/repeat",
        s=20,
        alpha=0.7,
    )
    ax.set_xlabel("error_model_0")
    ax.set_ylabel(f"error_model_{i}")
    plt.show()


def get_cur_pred_error(pred_df, i, row):
    mask = (pred_df[params] == row[params]).all(axis=1)
    cur_preds = pred_df[mask].reset_index(drop=True)

    label = f"error_model_{i}"
    cur_preds[label] = cur_preds["true"] - cur_preds["pred_stage3_total"]

    cur_preds_sub = cur_preds[["DAWBA_ID", "repeat", "split_id", label]].set_index(
        ["DAWBA_ID", "repeat", "split_id"]
    )
    return cur_preds_sub


def plot_cluster(compare_df, pred_df, prob_threshold):
    inconclusive_mask = (
        compare_df[["P(best better)", "P(equal)", "P(this better)"]] < prob_threshold
    ).all(axis=1)
    conclusive_mask = compare_df["P(equal)"] > prob_threshold
    eq_param_df = compare_df[inconclusive_mask | conclusive_mask]

    display(eq_param_df)
    if len(eq_param_df) == 1:
        print("NO models do as well, returning...")
        return None, eq_param_df
    model_frames = []
    for i, (idx, row) in enumerate(eq_param_df.iterrows()):
        cur_preds_sub = get_cur_pred_error(pred_df, idx, row)
        model_frames.append(cur_preds_sub)
        if i != 0:
            plot_scatter(model_frames[0], cur_preds_sub, idx)

    residual_wide = pd.concat(model_frames, axis=1, join="outer")  # instances x models
    residual_corr = (
        residual_wide.corr()
    )  # pairwise-complete by default, robust to misaligned folds
    sns.clustermap(
        residual_corr, cmap="viridis", annot=True, fmt=".2f", vmin=0.97, vmax=1
    )
    return residual_corr, eq_param_df

In [ ]:
num_layers = 2
impact_type = "parent"
two_layer_base = "Q"
pred_df = parent_base_df if impact_type == "parent" else self_base_df
if two_layer_base == None:
    imp_path = compare_dir / f"{num_layers}/{impact_type}_{SCORE_TYPE}.parquet"
elif num_layers == 2:
    imp_path = compare_dir / f"2/{impact_type}_{SCORE_TYPE}_{two_layer_base}.parquet"
compare_df = pd.read_parquet(imp_path)
residual_corr, eq_param = plot_cluster(compare_df, pred_df, prob_threshold=0.5)

In [ ]:
BEST_PARAM_DICT = {
    "SQT": {
        "parent": {
            "agg_type": "scale-new",
            "llm_model": "Qwen3.6-27B",
            "desc_seg": "shot_zero",
            "alpha": "15.0_75.0",
            "temp": 0.0,
            "num_layers": 3,
            "two_layer_base": None,
        },
        "self": {
            "agg_type": "scale-new",
            "llm_model": "gemini-2.5-pro",
            "desc_seg": "shot_few",
            "alpha": "50.0_15.0",
            "temp": 0.4,
            "num_layers": 3,
            "two_layer_base": None,
        },
    },
    "ST": {
        "parent": {
            "agg_type": "scale-new",
            "llm_model": "Llama-3.3-70B-Instruct",
            "desc_seg": "shot_few",
            "alpha": "15.0",
            "temp": 0.0,
            "num_layers": 2,
            "two_layer_base": "S",
        },
        "self": {
            "agg_type": "scale-new",
            "llm_model": "gemini-2.5-pro",
            "desc_seg": "shot_few",
            "alpha": "25.0",
            "temp": 0.4,
            "num_layers": 2,
            "two_layer_base": "S",
        },
    },
    "QT": {
        "parent": {
            "agg_type": "scale-new",
            "llm_model": "Qwen3.6-27B",
            "desc_seg": "shot_zero",
            "alpha": "5.0_50.0",
            "temp": 0.0,
            "num_layers": 2,
            "two_layer_base": "Q",
        },
        "self": {
            "agg_type": "scale-new",
            "llm_model": "gemini-2.5-pro",
            "desc_seg": "shot_few",
            "alpha": "15.0_5.0",
            "temp": 0.4,
            "num_layers": 2,
            "two_layer_base": "Q",
        },
    },
    "T": {
        "parent": {
            "agg_type": "scale-new",
            "llm_model": "Llama-3.3-70B-Instruct",
            "desc_seg": "shot_few",
            "alpha": "5.0",
            "temp": 1.0,
            "num_layers": 1,
            "two_layer_base": None,
        },
        "self": {
            "agg_type": "scale-new",
            "llm_model": "gemini-2.5-pro",
            "desc_seg": "shot_few",
            "alpha": "5.0",
            "temp": 0.4,
            "num_layers": 1,
            "two_layer_base": None,
        },
    },
}

### Calculate Bayesian compare vals- Q vs T for preliminary indication

- SQ vs ST
- Q vs T

#### Compare

In [ ]:
def compare_st_sq(r2_df, pred_df, best_T_params, score_type="r2"):
    r2_3_layer = get_layer3(r2_df, filter_best_Q=True)
    assert r2_3_layer["r2_Q_mean"].nunique() == 1

    # SQ scores
    best_Q_alpha = r2_3_layer.iloc[0]["alpha"].split("_")[0]
    mask = (pred_df["alpha"].str.split("_").str[0] == best_Q_alpha) & (
        pred_df["num_layers"] == 3
    )
    best_Q_pred_df = pred_df[mask].drop_duplicates(
        subset=["DAWBA_ID", "repeat", "split_id"]
    )
    print(best_Q_pred_df.shape)
    best_q_score_repeats_folds = (
        best_Q_pred_df.groupby(["repeat", "split_id"])
        .apply(lambda g: r2_score(y_pred=g["pred_stage2_total"], y_true=g["true"]))
        .rename("r2")
        .reset_index()
    )

    ## ST scores
    (
        score_df_fold_repeats,
        _,
        _,
        _,
    ) = get_score(pred_df, score_type, param_vals=best_T_params)

    ## Combine + compare
    suffix_1 = "_ST"
    suffix_2 = "_SQ"
    combined_df = pd.merge(
        score_df_fold_repeats,
        best_q_score_repeats_folds,
        on=["repeat", "split_id"],
        suffixes=(suffix_1, suffix_2),
    )
    (p_left, p_rope, p_right), ax = two_on_single(
        x=combined_df[f"{score_type}{suffix_1}"],
        y=combined_df[f"{score_type}{suffix_2}"],
        rope=0.01,
        runs=10,
        plot=True,
    )
    return p_left, p_rope, p_right, combined_df

In [ ]:
p_left_p, p_rope_p, p_right_p, compare_df_2_p = compare_st_sq(
    r2_df=parent_r2,
    pred_df=parent_base_df,
    best_T_params=pd.Series(BEST_PARAM_DICT["ST"]["parent"]),
)
print(f"Prob ST > SQ: {p_left_p:.1%}")
print(f"Prob ST = SQ: {p_rope_p:.1%}")
print(f"Prob ST < SQ: {p_right_p:.1%}")

In [ ]:
p_left_s, p_rope_s, p_right_s, compare_df_2_s = compare_st_sq(
    r2_df=self_r2,
    pred_df=self_base_df,
    best_T_params=pd.Series(BEST_PARAM_DICT["ST"]["self"]),
)
print(f"Prob ST > SQ: {p_left_s:.1%}")
print(f"Prob ST = SQ: {p_rope_s:.1%}")
print(f"Prob ST < SQ: {p_right_s:.1%}")

In [ ]:
import numpy as np


###########################
###########################
###########################
def compare_S_Q_T(r2_df, pred_df, score_type, best_T_params):
    # S scores
    S_pred_df = pred_df[pred_df["num_layers"] == 1].drop_duplicates(
        subset=["DAWBA_ID", "pred_stage1", "repeat", "split_id"]
    )
    score_df_fold_repeats_s = (
        S_pred_df.groupby(["repeat", "split_id"])
        .apply(lambda g: r2_score(y_pred=g["pred_stage1"], y_true=g["true"]))
        .rename("r2")
        .reset_index()
    )
    score_df_repeats_s = (
        S_pred_df.groupby(["repeat"])
        .apply(lambda g: r2_score(y_pred=g["pred_stage1"], y_true=g["true"]))
        .rename("r2")
        .reset_index()
    )
    # Q scores
    r2_1_layer = get_layer1(r2_df)
    best_Q_alpha = (
        r2_1_layer.sort_values(by="r2_Q_mean", ascending=False)
        .iloc[0]["alpha"]
        .split("_")[0]
    )
    q_mask = (pred_df["alpha"] == best_Q_alpha) & (pred_df["num_layers"] == 1)
    best_Q_pred_df = pred_df[q_mask].drop_duplicates(
        subset=["DAWBA_ID", "repeat", "split_id"]
    )

    # No need to pass in params bs already subset
    (
        _,
        _,
        score_df_fold_repeats_q,
        score_df_repeats_q,
    ) = get_score(best_Q_pred_df, score_type)

    # T scores
    (
        score_df_fold_repeats_t,
        score_df_repeats_t,
        _,
        _,
    ) = get_score(pred_df, score_type, param_vals=best_T_params)

    compare_dict = {
        "S": {
            "repeats/folds": score_df_fold_repeats_s,
            "repeats": score_df_repeats_s,
        },
        "Q": {
            "repeats/folds": score_df_fold_repeats_q,
            "repeats": score_df_repeats_q,
        },
        "T": {
            "repeats/folds": score_df_fold_repeats_t,
            "repeats": score_df_repeats_t,
        },
    }
    ## Make comparison table
    models = list(compare_dict)  # ['S', 'Q', 'T']
    p_better = pd.DataFrame(index=models, columns=models, dtype=float)  # P(row > col)
    p_equal = pd.DataFrame(index=models, columns=models, dtype=float)  # P(row = col)
    model_scores = pd.DataFrame(
        index=models,
        columns=["avg across repeats/folds (std)", "avg within repeats (std)"],
    )  # P(row = col)
    for a in models:
        rf = compare_dict[a]["repeats/folds"][score_type]
        rp = compare_dict[a]["repeats"][score_type]
        model_scores.loc[a, "avg across repeats/folds (std)"] = (
            f"{rf.mean():.3f} ({rf.std():.3f})"
        )
        model_scores.loc[a, "avg within repeats (std)"] = (
            f"{rp.mean():.3f} ({rp.std():.3f})"
        )
        for b in models:
            if a == b:
                p_better.loc[a, b] = np.nan  # diagonal: self-comparison undefined
                p_equal.loc[a, b] = 1.0
                continue

            merged = pd.merge(
                compare_dict[a]["repeats/folds"],
                compare_dict[b]["repeats/folds"],
                on=["repeat", "split_id"],
                suffixes=(f"_{a}", f"_{b}"),
            )
            p_left, p_rope, p_right = two_on_single(
                x=merged[f"{score_type}_{a}"],
                y=merged[f"{score_type}_{b}"],
                rope=0.01,
                runs=10,
            )
            p_better.loc[a, b] = p_left * 100  # P(a > b)
            p_equal.loc[a, b] = p_rope * 100  # P(a = b)

    for a in models:
        for b in models:
            if a != b:
                total = p_better.loc[a, b] + p_equal.loc[a, b] + p_better.loc[b, a]
                assert abs(total - 100) < 1e-6, f"{a},{b}: {total}"
    return p_better, p_equal, model_scores

In [ ]:
p_better_p, p_equal_p, model_scores_p = compare_S_Q_T(
    r2_df=parent_r2,
    pred_df=parent_base_df,
    score_type="r2",
    best_T_params=pd.Series(BEST_PARAM_DICT["T"]["parent"]),
)
print("P(row > col):\n", p_better_p.round(1))
print("\nP(row = col):\n", p_equal_p.round(1))
print("Model scores:\n", model_scores_p)

In [ ]:
p_better_s, p_equal_s, model_scores_s = compare_S_Q_T(
    r2_df=self_r2,
    pred_df=self_base_df,
    score_type="r2",
    best_T_params=pd.Series(BEST_PARAM_DICT["T"]["self"]),
)
print("P(row > col):\n", p_better_s.round(1))
print("\nP(row = col):\n", p_equal_s.round(1))
print("Model scores:\n", model_scores_s)